<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/Implementing%20a%20RESTful%20API%20for%20MultiSystem%20Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install fastapi uvicorn pydantic pytest httpx

In [16]:
pip install 'pydantic[email]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.7 MB/s eta 0:00:00


In [2]:
from fastapi import FastAPI,HTTPException,Depends,Security
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel,Field
from typing import Optional
import uuid

In [4]:
#1.Authentication
API_KEY = "super-secret-bookstore-key"
api_key_header = APIKeyHeader(name="X-API-Key",auto_error=False)

async def verify_api_key(api_key: str = Security(api_key_header)):
  if api_key != API_KEY:
    raise HTTPException(status_code=403,detail ="Invalid or missing API Key")
  return api_key

In [12]:
#2.Mock Subsystems(In-Memory Storage)
class MockInventoryDB:
    def __init__(self):
        self.books ={
            "1":{"book_id":"1","title":"1984","author":"George Orwell","price":15.99,"stock":100},
            "2":{"book_id":"2","title":"Dune","author":"Frank Herbert","price":18.50,"stock":50}

        }

class MockSalesDB:
    def __init__(self):
        self.orders = {}

class MockDeliveryDB:
    def __init__(self):
        self.dispatches = {}

inventory_db = MockInventoryDB()
sales_db = MockSalesDB()
delivery_db = MockDeliveryDB()

#3.Pydantic Models (Data Validation)
class OrderRequest(BaseModel):
    order_id: str
    shipping_address: str
    courier: str

class DeliveryStatusUpdate(BaseModel):
    status: str

In [17]:
from pydantic import BaseModel,Field, EmailStr

class OrderCreationRequest(BaseModel):
    book_id: str
    quantity: int = Field(..., gt=0)
    customer_email: EmailStr

In [14]:
#4.FastAPI APP & Endpoints
app = FastAPI(
    title = "Bookstore Management API",
    description ="API intergrating Inventory,Sales,and Delivery subsystems.",
    version="1.0.0"
)

#---INVENTORY SYSTEM--
@app.get("/api/v1/inventory/books/{book_id}",tags=["Inventory"])
def get_book_details(book_id:str,api_key:str = Depends(verify_api_key)):
  book = inventory_db.books.get(book_id)
  if not book:
    raise HTTPException(status_code = 404,detail='Book not found in inventory')
  # This line references 'order' which is not defined in this endpoint's scope.
  # It should likely be part of an order creation endpoint.
  # if book["stock"] < order.quantity: # Changed to lowercase 'stock' for consistency
  #   raise HTTPException(status_code=400, detail="Insufficient stock")
  return book

# The following code blocks ('Process Payment' and 'Save Order') are currently not
# part of any function definition and would execute directly. They likely belong
# within a new FastAPI endpoint, e.g., for creating an order.

# #2.Process Payment (Mock) & Deduct Stock
# book["stock"] -= order.quantity
# total_price = book["price"] * order.quantity
# payment_id = f"pay_{uuid.uuid4().hex[:8]}"

# #3.Save Order
# order_id = f"ord_{uuid.uuid4().hex[:8]}"
# order_record = {
#     "order-id":order_id,
#     "book_id":order.book_id,
#     "quantity":order.quantity,
#     "customer_email":order.customer_email,
#     "total_price":total_price,
#     "status":"PAID",
#     "payment_id":payment_id
# }
# sales_db.orders[order_id] = order_record
# return order_record


#--DELIVERY SYSTEM ---
@app.post("/api/v1/delivery/disputes",tags=["Delivery"],status_code=201)
def create_dispatch(dispatch:OrderRequest,api_key:str=Depends(verify_api_key)): # Assuming DeliveryRequest should be OrderRequest for now
  #Verify order exists and is paid
  order = sales_db.orders.get(dispatch.order_id)
  if not order:
    raise HTTPException(status_code=404,detail="Order not found")
  if order["status"] != "PAID":
    raise HTTPException(status_code=400,detail="Order must be paid before dispatch")

  #2.Create Delivery Record
  delivery_id = f"del_{uuid.uuid4().hex[:8]}"
  tracking_number = f"TRK-{uuid.uuid4().hex[:6].upper()}"

  delivery_record = {
      "delivery_id":delivery_id,
      "order_id":dispatch.order_id,
      "shipping_address":dispatch.shipping_address, # Corrected typo
      "courier":dispatch.courier,
      "status":"DISPATCHED",
      "tracking_number":tracking_number

  }
  delivery_db.dispatches[delivery_id] = delivery_record

  #3.Update Sales System Status
  sales_db.orders[dispatch.order_id]["status"] = "SHIPPED"

  return delivery_record # Corrected variable name

@app.patch("/api/v1/delivery/dispatches/{delivery_id}",tags=["Delivery"]) # Corrected path parameter
def update_delivery_status(delivery_id:str,update:DeliveryStatusUpdate,api_key:str=Depends(verify_api_key)):

   dispatch = delivery_db.dispatches.get(delivery_id)
   if not dispatch:
    raise HTTPException(status_code=404,detail="Delivery not found")

   if update.status == "Delivered": # Corrected indentation and added colon

   # Update order status in Sales system
     sales_db.orders[dispatch["order_id"]]["status"] = "DELIVERED"

     return dispatch

   dispatch["status"] = update.status # Update delivery status in dispatch record
   return dispatch

#---SALES SYSTEM---

In [13]:
@app.post("/api/v1/sales/orders",tags=["Sales"],status_code=201)
def create_order(order_details:OrderCreationRequest, api_key:str = Depends(verify_api_key)):
  #1.Check Inventory
  book = inventory_db.books.get(order_details.book_id)
  if not book:
    raise HTTPException(status_code=404,detail="Book not found in inventory")
  if book["stock"] < order_details.quantity:
    raise HTTPException(status_code=400,detail="Insufficient stock")

  #2.Process Payment (Mock) & Deduct Stock
  book["stock"] -= order_details.quantity # Deduct from inventory
  total_price = book["price"] * order_details.quantity
  payment_id = f"pay_{uuid.uuid4().hex[:8]}"

  #3.Save Order
  order_id = f"ord_{uuid.uuid4().hex[:8]}"
  order_record = {
      "order_id":order_id,
      "book_id":order_details.book_id,
      "quantity":order_details.quantity,
      "customer_email":order_details.customer_email,
      "total_price":total_price,
      "status":"PAID",
      "payment_id":payment_id
  }
  sales_db.orders[order_id] = order_record

  return order_record